In [1]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 7.1 MB/s  0:00:00 eta 0:00:01m


In [3]:
import psycopg2

conn = psycopg2.connect(host="localhost",
                        database= "Stock_database",
                        user="postgres",
                        password="1234",
                        port="5432")

In [4]:
cur = conn.cursor()

In [7]:
import sys
import os

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, project_root)

from backend_handlers.record_feeders.news.final_accumulator import build_news_dataframe

In [ ]:
company_slug = "reliance-industries"

df = build_news_dataframe(company_slug=company_slug, keywords=["reliance", "jio", "ril"], ticker="RELIANCE.NS")

Scraping page 1 - found 8 articles
Scraping page 2 - found 26 articles
Scraping page 3 - found 26 articles
Total collected: 7
Fetched 1 entries from Moneycontrol
Fetched 50 entries from Economic Times
Fetched 35 entries from LiveMint
Fetched 0 entries from Business Standard


In [14]:
df["company"] = company_slug

In [15]:
df.head()

,title,url,source,published,scraped_at,company
7,Reliance Industries Ltd stock (INE002A01018): ...,https://news.google.com/rss/articles/CBMiuwFBV...,Google News,"Wed, 15 Apr 2026 20:58:00 GMT",2026-04-18T21:00:32.416960,reliance-industries
34,Dividend Stock 2026: Reliance Industries-backe...,https://news.google.com/rss/articles/CBMi9AFBV...,Google News,"Wed, 15 Apr 2026 15:04:59 GMT",2026-04-18T21:00:34.157680,reliance-industries
32,India Stocks Soar: Global Optimism Lifts Marke...,https://news.google.com/rss/articles/CBMi0AFBV...,Google News,"Wed, 15 Apr 2026 03:59:09 GMT",2026-04-18T21:00:34.157676,reliance-industries
35,PPFAS Portfolio Churn: Rajeev Thakkar-led fund...,https://news.google.com/rss/articles/CBMilgJBV...,Google News,"Thu, 16 Apr 2026 09:40:49 GMT",2026-04-18T21:00:34.157682,reliance-industries
8,"Reliance Industries shares slump nearly 4%, ma...",https://news.google.com/rss/articles/CBMi5wFBV...,Google News,"Sun, 05 Apr 2026 07:00:00 GMT",2026-04-18T21:00:32.416963,reliance-industries


In [20]:
import pandas as pd

df["date"] = pd.to_datetime(
    df["published"],
    format="mixed",     # handles multiple formats
    errors="coerce",    # avoids crashes
    utc=True            # standardize timezone
)

# Extract year, month, day
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

df.head()

,title,url,source,published,scraped_at,company,date,year,month,day
7,Reliance Industries Ltd stock (INE002A01018): ...,https://news.google.com/rss/articles/CBMiuwFBV...,Google News,"Wed, 15 Apr 2026 20:58:00 GMT",2026-04-18T21:00:32.416960,reliance-industries,2026-04-15 20:58:00+00:00,2026,4,15
34,Dividend Stock 2026: Reliance Industries-backe...,https://news.google.com/rss/articles/CBMi9AFBV...,Google News,"Wed, 15 Apr 2026 15:04:59 GMT",2026-04-18T21:00:34.157680,reliance-industries,2026-04-15 15:04:59+00:00,2026,4,15
32,India Stocks Soar: Global Optimism Lifts Marke...,https://news.google.com/rss/articles/CBMi0AFBV...,Google News,"Wed, 15 Apr 2026 03:59:09 GMT",2026-04-18T21:00:34.157676,reliance-industries,2026-04-15 03:59:09+00:00,2026,4,15
35,PPFAS Portfolio Churn: Rajeev Thakkar-led fund...,https://news.google.com/rss/articles/CBMilgJBV...,Google News,"Thu, 16 Apr 2026 09:40:49 GMT",2026-04-18T21:00:34.157682,reliance-industries,2026-04-16 09:40:49+00:00,2026,4,16
8,"Reliance Industries shares slump nearly 4%, ma...",https://news.google.com/rss/articles/CBMi5wFBV...,Google News,"Sun, 05 Apr 2026 07:00:00 GMT",2026-04-18T21:00:32.416963,reliance-industries,2026-04-05 07:00:00+00:00,2026,4,5


In [21]:
df_final = df[["company", "title", "url", "year", "month", "day"]]

In [23]:
df_final.head()

,company,title,url,year,month,day
7,reliance-industries,Reliance Industries Ltd stock (INE002A01018): ...,https://news.google.com/rss/articles/CBMiuwFBV...,2026,4,15
34,reliance-industries,Dividend Stock 2026: Reliance Industries-backe...,https://news.google.com/rss/articles/CBMi9AFBV...,2026,4,15
32,reliance-industries,India Stocks Soar: Global Optimism Lifts Marke...,https://news.google.com/rss/articles/CBMi0AFBV...,2026,4,15
35,reliance-industries,PPFAS Portfolio Churn: Rajeev Thakkar-led fund...,https://news.google.com/rss/articles/CBMilgJBV...,2026,4,16
8,reliance-industries,"Reliance Industries shares slump nearly 4%, ma...",https://news.google.com/rss/articles/CBMi5wFBV...,2026,4,5


In [24]:
cur.execute("""
            CREATE TABLE IF NOT EXISTS news (
                id SERIAL PRIMARY KEY,
                company VARCHAR(255),
                title TEXT,
                url TEXT,
                year INT,
                month INT,
                day INT
            )
            """)

In [ ]:
### Test transcripts

In [ ]:
from backend_handlers.record_feeders.transcripts.get_nse_filings import get_nse_filings
from backend_handlers.record_feeders.transcripts.extract_documents import extract_documents
filings_df = get_nse_filings("RELIANCE")
docs = extract_documents(filings_df)


In [32]:
data = docs  # your dictionary

filtered_data = {
    key: [item for item in value if item.get("url") and item.get("url") != "-"]
    for key, value in data.items()
}

In [39]:
data_cons = [filtered_data["transcript"][-1]] + filtered_data["results"][:3]

In [40]:
data_cons

[{'title': 'Transcript of Analysts/Institutional Investor Meet/Con. Call',
  'url': 'https://nsearchives.nseindia.com/corporate/RELIANCE_07052022113222_AVRecordingandTranscript.pdf'},
 {'title': 'Financial Result Updates',
  'url': 'https://nsearchives.nseindia.com/corporate/SE_Result_16012025192542.pdf'},
 {'title': 'Financial Result Updates',
  'url': 'https://nsearchives.nseindia.com/corporate/BM_SE_14102024190232.pdf'},
 {'title': 'Financial Result Updates',
  'url': 'https://nsearchives.nseindia.com/corporate/SEFR_19072024191604.pdf'}]

In [41]:
df_transcripts = pd.DataFrame(data_cons)

In [42]:
df_transcripts

,title,url
0,Transcript of Analysts/Institutional Investor ...,https://nsearchives.nseindia.com/corporate/REL...
1,Financial Result Updates,https://nsearchives.nseindia.com/corporate/SE_...
2,Financial Result Updates,https://nsearchives.nseindia.com/corporate/BM_...
3,Financial Result Updates,https://nsearchives.nseindia.com/corporate/SEF...


In [43]:
df_transcripts["company"] = "RELIANCE"

In [44]:
df_transcripts

,title,url,company
0,Transcript of Analysts/Institutional Investor ...,https://nsearchives.nseindia.com/corporate/REL...,RELIANCE
1,Financial Result Updates,https://nsearchives.nseindia.com/corporate/SE_...,RELIANCE
2,Financial Result Updates,https://nsearchives.nseindia.com/corporate/BM_...,RELIANCE
3,Financial Result Updates,https://nsearchives.nseindia.com/corporate/SEF...,RELIANCE


In [ ]:
from backend_handlers.record_feeders.transcripts.get_pdf import download_pdf



In [2]:
from backend_handlers.record_feeders.transcripts.transcript_final_executor import process_transcripts

df_transcripts_final = process_transcripts("RELIANCE")

Saved: transcripts/Con. Call_2026-04-19_1.pdf
Saved: transcripts/RELIANCE_Financial Result Updates_2026-04-19_2.pdf
Saved: transcripts/RELIANCE_Financial Result Updates_2026-04-19_3.pdf
Saved: transcripts/RELIANCE_Financial Result Updates_2026-04-19_4.pdf


In [3]:
df_transcripts_final.head()

,title,url,filepath,symbol,date
0,Transcript of Analysts/Institutional Investor ...,https://nsearchives.nseindia.com/corporate/REL...,transcripts//Con. Call_2026-04-19_1.pdf,RELIANCE,2026-04-19
1,Financial Result Updates,https://nsearchives.nseindia.com/corporate/SE_...,transcripts//RELIANCE_Financial Result Updates...,RELIANCE,2026-04-19
2,Financial Result Updates,https://nsearchives.nseindia.com/corporate/BM_...,transcripts//RELIANCE_Financial Result Updates...,RELIANCE,2026-04-19
3,Financial Result Updates,https://nsearchives.nseindia.com/corporate/SEF...,transcripts//RELIANCE_Financial Result Updates...,RELIANCE,2026-04-19


In [ ]:
cur.execute("""
            CREATE TABLE IF NOT EXISTS transcripts_2 (
                id SERIAL PRIMARY KEY,
                company VARCHAR(255),
                title TEXT,
                url TEXT,
                filepath TEXT,
                date DATE
            )
            """)

In [ ]:

import requests
import time
url = f"https://www.nseindia.com/api/equity-stockIndices?index=NIFTY%20IT"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/",
}

session = requests.Session()
session.headers.update(headers)

for attempt in range(2):
    # Visit homepage first to establish session and get cookies
    home_response = session.get(
        "https://www.nseindia.com", 
    )
    print(home_response.status_code)
    
    # Longer delay to ensure cookies are set
    time.sleep(1.5)

    # Fetch the actual index data
    response = session.get(url)
    response.raise_for_status()

    data = response.json()
    


200
{'name': 'NIFTY IT', 'advance': {'declines': '3', 'advances': '7', 'unchanged': '0'}, 'timestamp': '17-Apr-2026 16:00:00', 'data': [{'priority': 1, 'symbol': 'NIFTY IT', 'identifier': 'NIFTY IT', 'open': 31656.55, 'dayHigh': 31959.35, 'dayLow': 31411.75, 'lastPrice': 31809.85, 'previousClose': 31817.5, 'change': -7.650000000001455, 'pChange': -0.02, 'ffmc': 122894437.05, 'yearHigh': 40301.4, 'yearLow': 28288.05, 'totalTradedVolume': 105098944, 'stockIndClosePrice': 0, 'totalTradedValue': 59213985135.75, 'lastUpdateTime': '17-Apr-2026 16:00:00', 'nearWKH': 21.07011170827813, 'nearWKL': -12.449780030790386, 'perChange365d': -4.68, 'perChange30d': 10.6, 'date365dAgo': '16-Apr-2025', 'date30dAgo': '17-Mar-2026', 'chartTodayPath': 'https://nsearchives.nseindia.com/today/NIFTY-IT.svg', 'chart30dPath': 'https://nsearchives.nseindia.com/30d/NIFTY-IT.svg', 'chart365dPath': 'https://nsearchives.nseindia.com/365d/NIFTY-IT.svg'}, {'priority': 0, 'symbol': 'WIPRO', 'identifier': 'WIPROEQN', 'se

In [68]:
data["data"]

[{'priority': 1,
  'symbol': 'NIFTY IT',
  'identifier': 'NIFTY IT',
  'open': 31656.55,
  'dayHigh': 31959.35,
  'dayLow': 31411.75,
  'lastPrice': 31809.85,
  'previousClose': 31817.5,
  'change': -7.650000000001455,
  'pChange': -0.02,
  'ffmc': 122894437.05,
  'yearHigh': 40301.4,
  'yearLow': 28288.05,
  'totalTradedVolume': 105098944,
  'stockIndClosePrice': 0,
  'totalTradedValue': 59213985135.75,
  'lastUpdateTime': '17-Apr-2026 16:00:00',
  'nearWKH': 21.07011170827813,
  'nearWKL': -12.449780030790386,
  'perChange365d': -4.68,
  'perChange30d': 10.6,
  'date365dAgo': '16-Apr-2025',
  'date30dAgo': '17-Mar-2026',
  'chartTodayPath': 'https://nsearchives.nseindia.com/today/NIFTY-IT.svg',
  'chart30dPath': 'https://nsearchives.nseindia.com/30d/NIFTY-IT.svg',
  'chart365dPath': 'https://nsearchives.nseindia.com/365d/NIFTY-IT.svg'},
 {'priority': 0,
  'symbol': 'WIPRO',
  'identifier': 'WIPROEQN',
  'series': 'EQ',
  'open': 205,
  'dayHigh': 206.45,
  'dayLow': 202.5,
  'lastPri

In [72]:


df = pd.DataFrame(data["data"])

# ✅ Remove index row (priority = 1)
df = df[df["priority"] == 0]

# ✅ Keep only required columns safely
required_cols = ["symbol", "lastPrice", "pChange"]
df = df[[col for col in required_cols if col in df.columns]]


# ✅ Convert to numeric (important for sorting)
df["pChange"] = pd.to_numeric(df["pChange"], errors="coerce")
df["lastPrice"] = pd.to_numeric(df["lastPrice"], errors="coerce")

# Drop NaNs just in case
df = df.dropna(subset=["pChange", "lastPrice"])

# ✅ Get top gainers
gainers = (
    df.sort_values("pChange", ascending=False)
      .head(3)
      .reset_index(drop=True)
)



In [74]:
from backend_handlers.record_feeders.market_positions.fetch_nse_index_gainers import fetch_nse_index

fetch_nse_index("NIFTY%20IT")

200


[{'priority': 1,
  'symbol': 'NIFTY IT',
  'identifier': 'NIFTY IT',
  'open': 31656.55,
  'dayHigh': 31959.35,
  'dayLow': 31411.75,
  'lastPrice': 31809.85,
  'previousClose': 31817.5,
  'change': -7.650000000001455,
  'pChange': -0.02,
  'ffmc': 122894437.05,
  'yearHigh': 40301.4,
  'yearLow': 28288.05,
  'totalTradedVolume': 105098944,
  'stockIndClosePrice': 0,
  'totalTradedValue': 59213985135.75,
  'lastUpdateTime': '17-Apr-2026 16:00:00',
  'nearWKH': 21.07011170827813,
  'nearWKL': -12.449780030790386,
  'perChange365d': -4.68,
  'perChange30d': 10.6,
  'date365dAgo': '16-Apr-2025',
  'date30dAgo': '17-Mar-2026',
  'chartTodayPath': 'https://nsearchives.nseindia.com/today/NIFTY-IT.svg',
  'chart30dPath': 'https://nsearchives.nseindia.com/30d/NIFTY-IT.svg',
  'chart365dPath': 'https://nsearchives.nseindia.com/365d/NIFTY-IT.svg'},
 {'priority': 0,
  'symbol': 'WIPRO',
  'identifier': 'WIPROEQN',
  'series': 'EQ',
  'open': 205,
  'dayHigh': 206.45,
  'dayLow': 202.5,
  'lastPri

In [3]:
from backend_handlers.record_feeders.market_positions.top_gainers_in_sector_nse import top_gainers_in_sector

top_gainers_in_sector("NIFTY%20IT")

200


,index,symbol,pChange
0,NIFTY%20IT,OFSS,2.90
1,NIFTY%20IT,TECHM,1.43
2,NIFTY%20IT,MPHASIS,0.67
3,NIFTY%20IT,COFORGE,0.36
4,NIFTY%20IT,TCS,0.26


In [ ]:
from backend_handlers.record_feeders.market_positions.market_postion_executor import get_market_positions

df =get_market_positions()

In [6]:
df.head()

,index,symbol,pChange,date
0,NIFTY%2050,HINDUNILVR,4.72,2026-04-19
1,NIFTY%2050,JSWSTEEL,2.20,2026-04-19
2,NIFTY%2050,NESTLEIND,2.20,2026-04-19
3,NIFTY%2050,APOLLOHOSP,2.08,2026-04-19
4,NIFTY%2050,POWERGRID,1.89,2026-04-19


In [80]:

cur.execute("""
            CREATE TABLE IF NOT EXISTS top_gainers (
                sector VARCHAR(255),
                company VARCHAR(255),
                pCHANGE FLOAT,
                date DATE
            )
            """)

In [8]:
from backend_handlers.database_utilities.table_creators import create 

create()

In [24]:
from backend_handlers.record_feeders.feed_top_gainers import feed_gainers

feed_gainers()

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
       sector     company  pCHANGE        date
0  NIFTY%2050       TRENT     3.86  2026-04-20
1  NIFTY%2050    JSWSTEEL     2.62  2026-04-20
2  NIFTY%2050        SBIN     2.49  2026-04-20
3  NIFTY%2050    ADANIENT     2.47  2026-04-20
4  NIFTY%2050  ASIANPAINT     1.91  2026-04-20
Inserted 135 rows into top_gainers


In [25]:
from backend_handlers.record_feeders.feed_news import feed_news

feed_news(company_slug='reliance-industries', keywords=["reliance", "jio", "ril"], ticker="RELIANCE.NS")

Scraping page 1 - found 8 articles
Scraping page 2 - found 26 articles
Scraping page 3 - found 26 articles
Total collected: 7
Fetched 1 entries from Moneycontrol
Fetched 50 entries from Economic Times
Fetched 35 entries from LiveMint
Fetched 0 entries from Business Standard
                company                                              title  \
38  reliance-industries  India Stocks Soar: Global Optimism Lifts Marke...   
35  reliance-industries  RIL shares drop 5% in 2 sessions: Why the stoc...   
22  reliance-industries  Stocks To Watch Today: HDFC Bank, Yes Bank, IC...   
33  reliance-industries  Reliance Q4 Results Date: RIL Plans FY26 Divid...   
21  reliance-industries  Stocks to Watch for April 20: HDFC Bank, ICICI...   

                                                  url  year  month  day  
38  https://news.google.com/rss/articles/CBMi0AFBV...  2026      4   15  
35  https://news.google.com/rss/articles/CBMizwFBV...  2026      4    7  
22  https://news.google.com/rss/ar

In [4]:
from backend_handlers.record_feeders.feed_transcripts import feed_transcripts

feed_transcripts("RELIANCE")

Saved: transcripts/Con. Call_2026-04-19_1.pdf
Saved: transcripts/RELIANCE_Financial Result Updates_2026-04-19_2.pdf
Saved: transcripts/RELIANCE_Financial Result Updates_2026-04-19_3.pdf
Saved: transcripts/RELIANCE_Financial Result Updates_2026-04-19_4.pdf
Inserted 4 rows into transcripts_2


In [28]:
from backend_handlers.database_utilities.read_database import read_table_to_dataframe
import datetime

top_gainers_df = read_table_to_dataframe("top_gainers", "Sector = 'NIFTY%20IT'")

def read_top_gainers(sector):
    date_today = datetime.datetime.now().strftime("%Y-%m-%d")
    print(date_today)
    #2026-04-19
    condition = f"Sector = '{sector.replace(' ', '%20')}' AND date = '{date_today}'"
    
    gainers_df = read_table_to_dataframe("top_gainers", condition)
    return gainers_df

/Users/abhijit/Desktop/Stock_Project/Stock_value_pred_NSE/backend_handlers/database_utilities/read_database.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [29]:
read_top_gainers("NIFTY IT")

2026-04-20


,sector,company,pchange,date
0,NIFTY%20IT,OFSS,1.80,2026-04-20
1,NIFTY%20IT,TCS,0.12,2026-04-20
2,NIFTY%20IT,TECHM,-0.02,2026-04-20
3,NIFTY%20IT,INFY,-0.23,2026-04-20
4,NIFTY%20IT,HCLTECH,-0.37,2026-04-20


In [11]:
top_gainers_df.head()

,sector,company,pchange,date
0,NIFTY%20IT,OFSS,2.90,2026-04-19
1,NIFTY%20IT,TECHM,1.43,2026-04-19
2,NIFTY%20IT,MPHASIS,0.67,2026-04-19
3,NIFTY%20IT,COFORGE,0.36,2026-04-19
4,NIFTY%20IT,TCS,0.26,2026-04-19


In [48]:
import datetime
from datetime import timedelta
from backend_handlers.database_utilities.execute_query import execute_query_to_dataframe

def read_news_from_database(company_slug):
    
    from datetime import datetime, timedelta

    date_str = (datetime.now() - timedelta(days=3)).strftime("%Y-%m-%d")
    query = f"""
    SELECT *
    FROM news
    WHERE company = '{company_slug}'
    AND MAKE_DATE(year, month, day) >= '{date_str}'
    """

    df = execute_query_to_dataframe(query)
    return df

In [49]:
read_news_from_database("reliance-industries")

/Users/abhijit/Desktop/Stock_Project/Stock_value_pred_NSE/backend_handlers/database_utilities/execute_query.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id,company,title,url,year,month,day
0,5,reliance-industries,"Q4 results 2026: Infosys, HCL Tech, Reliance I...",https://www.livemint.com/market/stock-market-n...,2026,4,19
1,6,reliance-industries,10 Sensex stocks with up to 45% upside potenti...,https://economictimes.indiatimes.com/markets/s...,2026,4,19
2,7,reliance-industries,Stocks to buy under ₹100: Sumeet Bagadia recom...,https://www.livemint.com/market/stock-market-n...,2026,4,19
3,8,reliance-industries,Buy or sell: Ganesh Dongre of Anand Rathi reco...,https://www.livemint.com/market/stock-market-n...,2026,4,19
4,9,reliance-industries,Q4 results 2026 to US-Iran war: Top five trigg...,https://www.livemint.com/market/stock-market-n...,2026,4,19
...,...,...,...,...,...,...,...
56,70,reliance-industries,Reliance Industries announces board meet date ...,https://news.google.com/rss/articles/CBMijgJBV...,2026,4,17
57,71,reliance-industries,Stock Market Today: Sensex Moves Up by 124 Poi...,https://news.google.com/rss/articles/CBMi2gFBV...,2026,4,17
58,72,reliance-industries,Reliance Power Share Price Rises 0.21% After M...,https://news.google.com/rss/articles/CBMiqgFBV...,2026,4,17
59,73,reliance-industries,Stock Market Today: शेअर बाजाराची सपाट सुरुवात...,https://news.google.com/rss/articles/CBMi3wFBV...,2026,4,17


In [64]:
import datetime
from datetime import timedelta
from backend_handlers.database_utilities.execute_query import execute_query_to_dataframe

def read_transcripts_from_database(company_slug):
    
    from datetime import datetime, timedelta

    query = f"""
    SELECT *
    FROM transcripts_2
    WHERE filepath = '{company_slug}'
    ORDER BY date DESC
    LIMIT 4
    """

    df = execute_query_to_dataframe(query)
    return df["url"].tolist()
 

In [65]:
read_transcripts_from_database("RELIANCE")

/Users/abhijit/Desktop/Stock_Project/Stock_value_pred_NSE/backend_handlers/database_utilities/execute_query.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


['transcripts//Con. Call_2026-04-19_1.pdf',
 'transcripts//RELIANCE_Financial Result Updates_2026-04-19_2.pdf',
 'transcripts//RELIANCE_Financial Result Updates_2026-04-19_3.pdf',
 'transcripts//RELIANCE_Financial Result Updates_2026-04-19_4.pdf']